**一、指令模板**

这是我们智能体的“说明书”，它将作为 ```system_prompt``` 传递给 LLM，告诉 LLM 它应该扮演什么角色、拥有哪些工具、以及如何格式化它的思考和行动。

In [ ]:
_user_preferences = {}

In [ ]:
AGENT_SYSTEM_PROMPT = """
你是一个智能旅行规划助手。你的任务是帮助用户规划完整的旅行，包括景点、交通、住宿等。你需要与用户交互，收集必要信息，然后使用可用工具一步步地获取数据，最终生成详细的行程计划。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气。
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。
- `recommend_accommodation(city, attractions, weather, preferences="")`: 查询指定城市的旅游者最多选择的居住地点（地铁站或景点附近）。
- `get_coordinate(place_name, city)`:查询景点和酒店的位置坐标。
- `get_travel_time_matrix(origins, destinations)`:查询两个地点之间行程耗时。
- `group_attractions_by_time(attractions, time_matrix, max_daily_travel=180)`:安排每日的行程安排，只包括每天去哪些景点。
- `plan_route(origin_coord, dest_coord, mode='transit', city='北京', fallback_to_driving=False)`: 获取两地之间的交通建议（如从酒店到景点）。
- `remember_preference(key: str, value: str)`: 记录用户的个性化偏好（如目的地、天数、兴趣、预算等），以便后续使用。
- `ask_user(question: str)`: 向用户提问以获取缺失的信息，例如目的地、旅行天数等。当你需要更多信息时使用此工具。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：不要添加任何其他内容。绝对不要在一条回复中输出多个 `Thought-Action` 对。

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action 的格式必须是以下之一：
1. 调用工具：function_name(arg_name=arg_value)
   - arg_value 可以是字符串（用双引号包裹）、数字、布尔值（True/False）、列表（用方括号）或字典（用花括号）。
   例如：get_attraction(city="北京", weather="晴天")
         group_attractions_by_time(attractions=["故宫", "天安门"], max_daily_travel=180)
2. 结束任务：Finish[最终答案]

# 重要提示:
- 你**必须**通过调用工具来获取所有信息，**绝对不允许**自己编造天气、景点、住宿等数据。
- 如果你试图不调用工具而直接输出行程，将导致严重错误。
- **每次调用工具后，你会收到一条 `Observation:` 结果。** 你必须基于最新的 `Observation:` 来思考下一步行动。
- 当你通过 `ask_user` 获取到用户信息后，必须立即使用 `remember_preference` 将其记录下来，例如：
  - 用户回答了预算：`remember_preference(key="budget", value="5000元/人")`
  - 用户回答了住宿偏好：`remember_preference(key="accommodation_pref", value="地铁沿线")`
- 在调用任何工具前，先检查 `_user_preferences`（通过 `remember_preference` 记录的键值对）中是否已有必要信息。如果缺少信息，优先使用 `ask_user` 提问。
- 如果你发现已经收集了所有必要信息（目的地、天数、兴趣、预算、住宿偏好等），并且已经获取了景点、天气、住宿推荐，就可以开始规划每日行程，最终输出完整的计划并用 `Finish` 结束。
- 绝对不要重复执行已经执行过的工具，除非有明确原因（如天气信息过期需刷新）。
- 如果你发现自己陷入了重复循环，请回顾最新的 `Observation`，确保你没有忽略已获得的信息。
- 你的每一步输出**必须**包含一对 `Thought:` 和 `Action:`，不要添加额外内容。且 `Action` 必须是工具调用或 `Finish[]`。
- Action必须在同一行，不要换行。
- get_travel_time_matrix(origins, destinations): 查询两个地点之间行程耗时。参数 origins 和 destinations 必须是 JSON 字符串，例如 origins="[[39.913951,116.410253],[39.917839,116.397029]]"。
- 每次回复只能包含一个 Thought 和一个 Action，绝不能输出多个，如果试图输出多个，将导致错误。
- 在调用 recommend_accommodation 之前，请务必先通过 remember_preference 记录景点和天气信息（键名分别为 'attractions' 和 'weather'），或者确保调用时传入了 attractions 和 weather 参数。
- 当收集到足够信息可以回答用户问题时，必须使用 Action: Finish[最终答案] 格式结束。
- 最终答案应该是一个详细的行程计划，包括每日的景点安排和交通方式。
请开始吧！
"""

**二、工具**

1.查询天气

我们将通过高德地图天气 API 查询指定城市的实时天气，并返回格式化的天气信息或错误提示。这里也可以使用其他工具，例如免费的天气查询服务 wttr.in ，它能以 JSON 格式返回指定城市的天气数据。下面是实现该工具的代码：

In [ ]:
import requests

def get_weather(city: str) -> str:
    """
    使用高德地图天气 API 查询实时天气。
    """
    api_key = os.getenv("GAODE_API_KEY")
    if not api_key:
        return "错误:未配置高德 API Key"
    
    url = "https://restapi.amap.com/v3/weather/weatherInfo"
    params = {
        "key": api_key,
        "city": city,
        "extensions": "base",
        "output": "json"
    }
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
        if data["status"] == "1" and data.get("lives"):
            live = data["lives"][0]
            weather = live["weather"]
            temperature = live["temperature"]
            return f"{city}当前天气:{weather}，气温{temperature}摄氏度"
        else:
            return f"错误:获取天气失败 - {data.get('info', '未知错误')}"
    except Exception as e:
        return f"错误:查询天气时出现异常 - {e}"




2.搜索并推荐旅游景点

我们将定义一个新工具 search_attraction ，它会基于给定的城市和天气条件，利用 Tavily 搜索引擎获取并返回该城市在特定天气下的旅游景点推荐。

In [ ]:
import os
from tavily import TavilyClient

def get_attraction(city: str, weather: str) -> str:
    """
    根据城市和天气，使用Tavily Search API搜索并返回优化后的景点推荐。
    """
    # 1. 从环境变量中读取API密钥
    api_key = os.environ.get("TAVILY_API_KEY")
    if not api_key:
        return "错误:未配置TAVILY_API_KEY环境变量。"

    # 2. 初始化Tavily客户端
    tavily = TavilyClient(api_key=api_key)

    # 3. 构造一个精确的查询
    query = f"'{city}' 在'{weather}'天气下最值得去的旅游景点推荐及理由"

    try:
        # 4. 调用API，include_answer=True会返回一个综合性的回答
        response = tavily.search(query=query, search_depth="basic",include_answer=True)
        # 5. Tavily返回的结果已经非常干净，可以直接使用
        # response['answer'] 是一个基于所有搜索结果的总结性回答
        if response.get("answer"):
            return response["answer"]

        # 如果没有综合性回答，则格式化原始结果
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")

        if not formatted_results:
            return "抱歉，没有找到相关的旅游景点推荐。"

        return "根据搜索，为您找到以下信息:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误:执行Tavily搜索时出现问题 - {e}"
        

3.查询酒店

下面我们将基于城市、景点、天气和用户偏好，通过大语言模型生成住宿区域推荐，代码如下：

In [ ]:
import os
from tavily import TavilyClient



def recommend_accommodation(city, attractions=None, weather=None, preferences="", llm_client=None):
    """
    根据城市、景点、天气和用户偏好，结合 Tavily 搜索结果，使用大语言模型推荐住宿区域。
    
    Args:
        city (str): 目的地城市。
        attractions (str, optional): 计划游览的景点列表（字符串形式）。若为None，则从全局偏好中获取。
        weather (str, optional): 当地天气描述。若为None，则从全局偏好中获取。
        preferences (str): 用户的其他要求（如预算、偏好类型等）。
        llm_client (object, optional): 大语言模型客户端，必须包含 generate(prompt, system_prompt) 方法。
    
    Returns:
        str: 住宿推荐结果或错误信息。
    """
    # 1. 参数获取与校验
    global _user_preferences

    if attractions is None:
        attractions = _user_preferences.get('attractions', '')
    if weather is None:
        weather = _user_preferences.get('weather', '')
    
    # 如果景点或天气信息缺失，无法进行合理推荐
    if not attractions or not weather:
        return "错误: 缺少景点或天气信息，无法推荐住宿。请先获取景点和天气数据。"
    
    # 2. 使用 Tavily 搜索相关信息
    tavily_api_key = os.getenv("TAVILY_API_KEY")
    if not tavily_api_key:
        return "错误: 未配置 TAVILY_API_KEY 环境变量。"
    
    try:
        tavily = TavilyClient(api_key=tavily_api_key)
    except Exception as e:
        return f"错误: 初始化 TavilyClient 失败 - {e}"
    
    # 构造搜索查询：结合城市、景点和住宿区域
    search_query = f"{city} 住宿 推荐 区域 地铁站 {attractions}"
    try:
        search_res = tavily.search(query=search_query, search_depth="basic", max_results=5)
    except Exception as e:
        return f"错误: Tavily 搜索失败 - {e}"
    
    # 3. 整理搜索结果作为上下文
    results = search_res.get("results", [])
    if results:
        context = "\n".join(
            f"- {r.get('title', '无标题')}: {r.get('content', '无内容')}" for r in results
        )
    else:
        context = "未找到相关网页信息。"
    
    # 4. 使用 LLM 生成推荐
    if llm_client is None:
        # 尝试获取全局变量中的 llm 对象
        llm_client = globals().get('llm')
        if llm_client is None:
            return "错误: 无法获取语言模型客户端。请提供 llm_client 参数或确保全局 llm 对象存在。"
    
    # 确保 llm_client 具有 generate 方法
    if not hasattr(llm_client, 'generate'):
        return "错误: 提供的 llm_client 对象没有 generate 方法。"
    
    prompt = f"""
你是一个经验丰富的旅游顾问。请基于以下搜索结果和用户信息，为用户推荐住宿区域（只需推荐地铁站或者景点附近，不用具体酒店名称），并解释为什么大多数人会选择那里。

用户信息：
- 目的地：{city}
- 计划游览景点：{attractions}
- 当地天气：{weather}
- 其他要求：{preferences}

搜索结果摘要：
{context}

请给出1个推荐的地铁站或区域，并简要说明理由。
"""
    try:
        response = llm_client.generate(prompt, system_prompt="你是一个旅游顾问，请简洁回答。")
        return response
    except Exception as e:
        return f"错误: 调用 LLM 生成推荐时发生异常 - {e}"

4.日程安排


首先，我们利用使用高德地图地理编码 API 将指定的地点名称（如景点、酒店名）转换为地理坐标（纬度和经度），代码如下：

In [ ]:
import requests
import time
def get_coordinate(place_name, city,api_key=None):
    """
    使用高德地图地理编码 API 查询地点坐标。
    优先使用传入的 api_key，若无效或未传入则尝试从环境变量获取。
    返回坐标元组 (纬度, 经度) 或错误字符串。
    """
    # 如果未传入 api_key 或传入的 key 无效（由后续调用判断），尝试从环境变量获取
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key，请在环境变量中设置 GAODE_API_KEY"
    # 注意：即使传入了 api_key，也可能无效，我们将在请求后统一处理错误

    url = "https://restapi.amap.com/v3/geocode/geo"
    params = {
        "key": api_key,
        "address": place_name,
        "city": city,
        "output": "json"
    }
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
    except Exception as e:
        return f"错误: 请求高德地理编码 API 时发生异常 - {e}"

    # 检查 API 返回状态
    if data.get("status") != "1":
        info = data.get("info", "未知错误")
        if info == "INVALID_USER_KEY":
            # 如果传入的 key 无效且不是从环境变量来的，可以尝试使用环境变量重试一次
            if api_key != os.getenv("GAODE_API_KEY") and os.getenv("GAODE_API_KEY"):
                print("传入的 API Key 无效，尝试使用环境变量中的 Key 重试...")
                return get_coordinate(place_name, city, api_key=os.getenv("GAODE_API_KEY"))
            return f"错误: 高德地理编码失败 - {info}，请检查 API Key 是否有效并已开通服务"
        else:
            return f"错误: 高德地理编码失败 - {info}"

    geocodes = data.get("geocodes")
    if not geocodes:
        return f"错误: 未找到地点 '{place_name}' 的坐标信息"

    location = geocodes[0].get("location")  # 格式 "经度,纬度"
    if not location:
        return f"错误: 地点 '{place_name}' 坐标数据为空"

    lng, lat = map(float, location.split(','))
    return (lng, lat)  # 返回 (经度, 纬度) 元组



有了坐标，你需要知道任意两个景点之间的交通时间（或距离），方便我们对景点按天进行分类。下面我们使用高德地图距离测量 API，计算多个起点到多个终点之间的驾车耗时，并以二维矩阵形式返回。

In [ ]:
def get_travel_time_matrix(origins, destinations, api_key=None):
    """
    查询多个起点到多个终点的驾车耗时矩阵。
    
    Args:
        origins (str): JSON 字符串，表示起点坐标列表，每个坐标为 [经度, 纬度] 的列表，例如 "[[116.411571,39.908069], [116.401304,39.905374]]"
        destinations (str): JSON 字符串，表示终点坐标列表，同样为经度,纬度格式
        api_key (str, optional): 高德 API Key，默认从环境变量获取
    
    Returns:
        str: 成功时返回耗时矩阵的 JSON 字符串（二维列表），失败时返回错误信息（以 "错误:" 开头）
    """
    import json
    import os
    import requests

    # 1. 解析 JSON 字符串为 Python 列表
    try:
        origins_list = json.loads(origins)
        destinations_list = json.loads(destinations)
    except json.JSONDecodeError as e:
        return f"错误: 解析 origins 或 destinations 失败 - {e}"
    
    # 2. 检查 API Key
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key"
    
    # 3. 构建高德距离矩阵 API 请求参数
    url = "https://restapi.amap.com/v3/distance"
    params = {
        "key": api_key,
        "type": "1",  # 1 表示驾车距离
        "origins": "|".join([f"{lng},{lat}" for lng, lat in origins_list]),  # 注意顺序：经度,纬度
        "destinations": "|".join([f"{lng},{lat}" for lng, lat in destinations_list]),
        "output": "json"
    }
    
    # 4. 发送请求
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
    except Exception as e:
        return f"错误: 请求高德距离矩阵 API 时发生异常 - {e}"
    
    # 5. 处理响应
    if data.get("status") != "1":
        return f"错误: 高德距离矩阵请求失败 - {data.get('info', '未知错误')}"
    
    results = data.get("results")
    if not results:
        return "错误: 返回结果为空"
    
    # 6. 将结果组装为二维耗时矩阵（秒）
    matrix = []
    num_dests = len(destinations_list)
    for i in range(len(origins_list)):
        row = []
        for j in range(num_dests):
            item = results[i * num_dests + j]
            row.append(float(item["duration"]))  # 耗时（秒）
        matrix.append(row)
    
    # 返回矩阵的 JSON 字符串
    return json.dumps(matrix)

下面我们使用简单的贪心分组法来安排每日行程，选择一个起始景点（例如住宿最近的那个），找出与它耗时最短的景点，
如果总耗时不超过每天合理游览时间（例如 3-4 小时交通+游览），就加入当天，重复直到当天无法再增加，然后开始下一天。

In [ ]:
def group_attractions_by_time(attractions, time_matrix, max_daily_travel=180):
    """
    根据景点间耗时矩阵，将景点分组到每天（贪心算法）。
    
    Args:
        attractions (str): JSON 字符串，表示景点名称列表，例如 '["中国国家博物馆","国家艺术博物馆"]'
        time_matrix (str): JSON 字符串，表示景点间耗时矩阵（秒），例如 '[[0,471],[471,0]]'
        max_daily_travel (int/str): 每天允许的最大交通耗时（分钟），默认 180 分钟。允许传入字符串，函数内部会自动转换。
    
    Returns:
        str: 成功时返回分组结果的 JSON 字符串（二维列表，每组为景点名称列表），失败时返回错误信息
    """
    import json

    # 1. 解析参数
    try:
        attractions_list = json.loads(attractions)
        time_matrix_list = json.loads(time_matrix)
    except json.JSONDecodeError as e:
        return f"错误: 解析 attractions 或 time_matrix 失败 - {e}"
    
    # 2. 处理 max_daily_travel（兼容字符串或数字）
    try:
        max_daily_travel = int(max_daily_travel)
    except (ValueError, TypeError):
        return f"错误: max_daily_travel 必须为整数，无法转换 '{max_daily_travel}'"
    
    # 3. 检查数据维度
    n = len(attractions_list)
    if not isinstance(time_matrix_list, list) or len(time_matrix_list) != n:
        return f"错误: time_matrix 维度不匹配，应为 {n}x{n} 的二维列表"
    for row in time_matrix_list:
        if not isinstance(row, list) or len(row) != n:
            return f"错误: time_matrix 维度不匹配，应为 {n}x{n} 的二维列表"
    
    # 4. 贪心分组算法
    visited = [False] * n
    groups = []
    
    max_daily_travel_sec = max_daily_travel * 60  # 转为秒
    
    for i in range(n):
        if visited[i]:
            continue
        # 开始新的一天
        day = [attractions_list[i]]
        visited[i] = True
        total_travel = 0
        current = i
        while True:
            # 找到与 current 耗时最短的未访问景点
            best_j = -1
            best_time = float('inf')
            for j in range(n):
                if not visited[j] and time_matrix_list[current][j] < best_time:
                    best_time = time_matrix_list[current][j]
                    best_j = j
            if best_j == -1:
                break
            # 如果加上这个耗时还允许，就加入当天
            if total_travel + best_time <= max_daily_travel_sec:
                day.append(attractions_list[best_j])
                visited[best_j] = True
                total_travel += best_time
                current = best_j
            else:
                break
        groups.append(day)
    
    # 返回分组结果的 JSON 字符串
    return json.dumps(groups, ensure_ascii=False)

下面的函数对路径进行规划，使用高德地图，选择合适的交通方式，包含耗时（分钟）、距离（公里）、路线。由于我未开通公交权限，无法使用公交规划，增加了一个辅助函数 ``plan_route_with_driving_fallback`` ，当 ```mode='transit'``` 且请求失败时，若启用了回退，则自动调用驾车模式重新规划，并在返回结果中添加 ``fallback_note`` 字段说明情况。

In [ ]:
def _format_route_description(route, mode):
    """
    根据交通模式，从 route 对象中提取信息并生成易读的路线描述。
    """
    def format_duration(seconds):
        minutes = seconds // 60
        if minutes < 60:
            return f"{minutes}分钟"
        else:
            hours = minutes // 60
            mins = minutes % 60
            return f"{hours}小时{mins}分钟" if mins else f"{hours}小时"

    def format_distance(meters):
        km = meters / 1000
        return f"{km:.1f}公里"

    if mode in ('driving', 'walking', 'cycling'):
        # 驾车、步行、骑行：取第一条路径
        paths = route.get('paths', [])
        if not paths:
            return "未找到可行路线。"
        path = paths[0]
        distance = float(path.get('distance', 0))
        duration = float(path.get('duration', 0))
        steps = path.get('steps', [])

        # 生成步骤简述（取前3个主要步骤）
        step_desc = []
        for step in steps[:3]:  # 只显示前3个步骤，避免过长
            instruction = step.get('instruction', '')
            if instruction:
                step_desc.append(instruction)
        if len(steps) > 3:
            step_desc.append("...")

        desc = f"全程约{format_distance(distance)}，预计耗时{format_duration(duration)}。"
        if step_desc:
            desc += " 路线：" + "；".join(step_desc)
        return desc

    elif mode == 'transit':
        # 公交模式：取第一条公交方案
        transits = route.get('transits', [])
        if not transits:
            return "未找到公交路线。"
        transit = transits[0]
        distance = float(transit.get('distance', 0))
        duration = float(transit.get('duration', 0))
        segments = transit.get('segments', [])

        # 生成步骤描述
        step_list = []
        for seg in segments:
            if 'walking' in seg:
                walk = seg['walking']
                steps = walk.get('steps', [])
                for step in steps[:1]:  # 步行取第一步说明
                    instruction = step.get('instruction', '步行')
                    step_list.append(instruction)
            elif 'bus' in seg:
                bus = seg['bus']
                buslines = bus.get('buslines', [])
                for bl in buslines[:1]:  # 只显示第一条公交线路
                    name = bl.get('name', '公交')
                    dep_stop = bl.get('departure_stop', {}).get('name', '')
                    arr_stop = bl.get('arrival_stop', {}).get('name', '')
                    if dep_stop and arr_stop:
                        step_list.append(f"乘坐{name}从{dep_stop}到{arr_stop}")
                    else:
                        step_list.append(f"乘坐{name}")
            elif 'railway' in seg:
                railway = seg['railway']
                name = railway.get('name', '火车')
                dep = railway.get('departure_stop', {}).get('name', '')
                arr = railway.get('arrival_stop', {}).get('name', '')
                step_list.append(f"乘坐{name}从{dep}到{arr}")

        # 限制步骤数量
        if len(step_list) > 5:
            step_list = step_list[:5] + ["..."]

        desc = f"公交方案：全程约{format_distance(distance)}，预计耗时{format_duration(duration)}。"
        if step_list:
            desc += " 步骤：" + "；".join(step_list)
        return desc

    else:
        return f"未知交通模式: {mode}"

In [ ]:
import requests

def plan_route(origin_coord, dest_coord, mode='transit', city='北京', fallback_to_driving=False, api_key=None):
    """
    高德路径规划API调用函数（支持公交不可用时自动回退到驾车）。
    返回易读的路线描述字符串，如果失败则返回错误信息。
    """
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            return "错误：未配置高德 API Key"

    origin = f"{origin_coord[0]},{origin_coord[1]}"
    destination = f"{dest_coord[0]},{dest_coord[1]}"

    fallback_used = False
    fallback_note = None

    # 如果启用回退且模式为公交，先尝试公交
    if mode == 'transit' and fallback_to_driving:
        result = _call_api(origin, destination, mode='transit', city=city, api_key=api_key)
        if result is not None:
            return result  # 成功，直接返回
        else:
            fallback_used = True
            fallback_note = "（公交规划不可用，已自动切换为驾车模式）"
            print(fallback_note)
            mode = 'driving'

    # 调用指定模式（可能是原模式，也可能是回退后的驾车）
    result = _call_api(origin, destination, mode=mode, city=city, api_key=api_key)
    if result is None:
        return f"错误：无法获取{mode}路线。"
    if fallback_used:
        result = fallback_note + " " + result
    return result

def _call_api(origin, destination, mode, city=None, api_key=None):
    """
    实际调用高德API的内部函数。
    成功时返回格式化的路线描述字符串，失败时返回None。
    """
    if api_key is None:
        api_key = os.getenv("GAODE_API_KEY")
        if api_key is None:
            print("错误：未配置高德 API Key")
            return None

    # 根据 mode 构建不同的 URL 和参数
    if mode == 'transit':
        url = "https://restapi.amap.com/v3/direction/integrated"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'city': city,
            'strategy': 0,
            'nightflag': 0,
            'output': 'json'
        }
    elif mode == 'driving':
        url = "https://restapi.amap.com/v3/direction/driving"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'strategy': 0,
            'extensions': 'all',
            'output': 'json'
        }
    elif mode == 'walking':
        url = "https://restapi.amap.com/v3/direction/walking"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'output': 'json'
        }
    elif mode == 'cycling':
        url = "https://restapi.amap.com/v3/direction/cycling"
        params = {
            'key': api_key,
            'origin': origin,
            'destination': destination,
            'output': 'json'
        }
    else:
        print(f"错误: 不支持的交通模式 '{mode}'")
        return None

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        if data.get('status') != '1':
            print(f"API错误 ({mode}): {data.get('info', '未知错误')}")
            return None

        route = data['route']
        # 根据 mode 解析并格式化
        return _format_route_description(route, mode)

    except Exception as e:
        print(f"请求异常 ({mode}): {e}")
        return None

In [ ]:
# 用于存储用户偏好的全局字典
#_user_preferences = {}

def remember_preference(key: str, value: str) -> str:
    """
    记录用户的个性化偏好（如目的地、天数、兴趣、预算等），以便后续使用。

    Args:
        key (str): 偏好项的名称，例如 'destination', 'days', 'interests', 'budget'。
        value (str): 偏好的具体值，由用户提供或从对话中提取。
    """
    global _user_preferences
    _user_preferences[key] = value
    # 可选：打印确认信息（便于调试）
    #print(f"已记录偏好：{key} = {value})
    return f"已记录偏好：{key} = {value}"

下面定义一个简单的 ask_user 函数，用于向用户提问并获取输入。该函数适用于命令行环境，通过 ```input()``` 获取用户回答，并返回清理后的字符串。

In [ ]:
def ask_user(question: str) -> str:
    """
    向用户提问以获取缺失的信息，例如目的地、旅行天数等。
    当你需要更多信息时使用此工具。

    Args:
        question (str): 要向用户提出的问题。

    Returns:
        str: 用户的回答（已去除首尾空白）。
    """
    print(question)
    answer = input().strip()
    return answer

In [ ]:
# 将所有工具函数放入一个字典，方便后续调用
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
    "recommend_accommodation":recommend_accommodation,
    "get_coordinate":get_coordinate,
    "get_travel_time_matrix":get_travel_time_matrix,
    "group_attractions_by_time":group_attractions_by_time,
    "plan_route":plan_route,
    "ask_user": ask_user,
    "remember_preference": remember_preference
}

In [ ]:
from openai import OpenAI

class OpenAICompatibleClient:
    """
    一个用于调用任何兼容OpenAI接口的LLM服务的客户端。
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """调用LLM API来生成回应。"""
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应成功。")
            return answer
        except Exception as e:
            print(f"调用LLM API时发生错误: {e}")
            return "错误:调用语言模型服务时出错。"

In [ ]:
import ast
import re

def parse_action(action_str):
    """
    解析 Action 字符串，返回 (工具名, 参数字典)
    例如：parse_action('get_weather(city="北京")') -> ('get_weather', {'city': '北京'})
    """
    # 提取工具名
    tool_match = re.match(r'(\w+)\s*\(', action_str)
    if not tool_match:
        raise ValueError(f"无法解析工具名: {action_str}")
    tool_name = tool_match.group(1)
    
    # 提取括号内的参数部分
    start = action_str.find('(') + 1
    end = action_str.rfind(')')
    if start == 0 or end == -1:
        raise ValueError(f"参数括号不完整: {action_str}")
    args_str = action_str[start:end].strip()
    
    # 解析参数
    kwargs = _parse_arguments(args_str)
    return tool_name, kwargs

def _parse_arguments(s):
    """解析参数字符串，返回字典"""
    if not s:
        return {}
    
    # 尝试作为单个字典参数解析（例如 {"city":"北京"}）
    if s.startswith('{') and s.endswith('}'):
        try:
            return ast.literal_eval(s)
        except:
            pass  # 不是合法字典，继续按命名参数解析
    
    kwargs = {}
    i = 0
    n = len(s)
    while i < n:
        # 跳过空白
        while i < n and s[i].isspace():
            i += 1
        if i >= n:
            break
        
        # 解析参数名（字母、数字、下划线）
        name_match = re.match(r'([_a-zA-Z][_a-zA-Z0-9]*)\s*=', s[i:])
        if not name_match:
            raise ValueError(f"期望参数名，但遇到: {s[i:i+20]}")
        name = name_match.group(1)
        i += name_match.end()  # 移动到 '=' 之后
        
        # 解析值
        value, i = _parse_value(s, i)
        kwargs[name] = value
        
        # 跳过空白，检查逗号
        while i < n and s[i].isspace():
            i += 1
        if i < n and s[i] == ',':
            i += 1
        # 继续下一个参数
    return kwargs

def _parse_value(s, start):
    """从索引 start 开始解析一个值，返回 (value, new_index)"""
    i = start
    n = len(s)
    # 跳过前导空格
    while i < n and s[i].isspace():
        i += 1
    if i >= n:
        raise ValueError("值未结束")
    
    ch = s[i]
    
    # 1. 字符串（单引号或双引号）
    if ch in ('"', "'"):
        quote = ch
        i += 1
        value_chars = []
        escaped = False
        while i < n:
            c = s[i]
            if escaped:
                value_chars.append(c)
                escaped = False
            elif c == '\\':
                escaped = True
            elif c == quote:
                i += 1  # 跳过结束引号
                break
            else:
                value_chars.append(c)
            i += 1
        else:
            raise ValueError("字符串未闭合")
        return ''.join(value_chars), i
    
    # 2. 数字（整数/浮点数）
    num_match = re.match(r'-?\d+(?:\.\d+)?', s[i:])
    if num_match:
        num_str = num_match.group()
        i += len(num_str)
        if '.' in num_str:
            return float(num_str), i
        else:
            return int(num_str), i
    
    # 3. 布尔值/None
    if s.startswith('True', i):
        return True, i + 4
    if s.startswith('False', i):
        return False, i + 5
    if s.startswith('None', i):
        return None, i + 4
    
    # 4. 列表或字典（使用 ast.literal_eval 安全求值）
    if ch in ('[', '{'):
        # 找到匹配的闭合括号
        stack = [ch]
        j = i + 1
        while j < n and stack:
            c = s[j]
            if c in ('[', '{'):
                stack.append(c)
            elif c == ']' and stack[-1] == '[':
                stack.pop()
            elif c == '}' and stack[-1] == '{':
                stack.pop()
            j += 1
        if stack:
            raise ValueError(f"{ch} 未闭合")
        try:
            val = ast.literal_eval(s[i:j])
        except:
            raise ValueError(f"无法解析列表/字典: {s[i:j]}")
        return val, j
    
    # 5. 其他情况（应不会发生）
    raise ValueError(f"无法识别的值开始字符: {ch}")



In [ ]:
import re
import os
import json

# --- 1. 配置LLM客户端 ---
# 请根据您使用的服务，将这里替换成对应的凭证和地址
API_KEY = "your_aihubmix_api_key"
BASE_URL = "https://aihubmix.com/v1"
MODEL_ID = "coding-glm-4.7-free" #可换别的
TAVILY_API_KEY="your_tavily_api_key"
os.environ['TAVILY_API_KEY'] = "your_tavily_api_key"
os.environ["GAODE_API_KEY"] = "your_gaode_api_key"

llm = OpenAICompatibleClient(
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL
)

# --- 2. 初始化 ---
print("\n请输入您的旅行需求（例如：我想去北京玩3天，喜欢历史文化）：")
user_prompt = input().strip()
prompt_history = [f"用户说：{user_prompt}"]
# --- 3. 运行主循环 ---
for i in range(30): # 设置最大循环次数
    print(f"--- 循环 {i+1} ---\n")
    # 3.1. 构建Prompt
    full_prompt = "\n".join(prompt_history)

    # 3.2. 调用LLM进行思考
    llm_output = llm.generate(full_prompt, system_prompt=AGENT_SYSTEM_PROMPT)
    # 模型可能会输出多余的Thought-Action，需要截断
    match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
    if match:
        truncated = match.group(1).strip()
        if truncated != llm_output.strip():
            llm_output = truncated
            print("已截断多余的 Thought-Action 对")
    print(f"模型输出:\n{llm_output}\n")
    prompt_history.append(llm_output)
    # 3.3. 解析并执行行动
        # 3.3. 解析并执行行动
    # 在循环内，解析 Action 后...
action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
if not action_match:
    observation = "错误: 未能解析到 Action 字段。请确保你的回复严格遵循 'Thought: ...Action: ...' 的格式。"
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)
    continue

action_str = action_match.group(1).strip()

if action_str.startswith("Finish"):
    final_match = re.search(r"Finish\[(.*)\]", action_str, re.DOTALL)
    if final_match:
        final_answer = final_match.group(1)
        print(f"任务完成，最终答案: {final_answer}")
        break
    else:
        observation = "错误: Finish 格式不正确，应为 Finish[最终答案]"
        observation_str = f"Observation: {observation}"
        print(f"{observation_str}\n" + "="*40)
        prompt_history.append(observation_str)
        continue

# ---------- 使用新解析器 ----------
try:
    tool_name, kwargs = parse_action(action_str)
except Exception as e:
    observation = f"错误: 解析 Action 参数时发生异常 - {e}"
    observation_str = f"Observation: {observation}"
    print(f"{observation_str}\n" + "="*40)
    prompt_history.append(observation_str)
    continue

# 调用工具
if tool_name in available_tools:
    try:
        print(f"正在调用工具 {tool_name}，参数: {kwargs}")
        observation = available_tools[tool_name](**kwargs)
        print(f"工具返回: {observation}")
    except TypeError as e:
        observation = f"错误: 调用工具 '{tool_name}' 时参数不匹配: {e}。提供的参数: {kwargs}"
    except Exception as e:
        observation = f"错误: 调用工具 '{tool_name}' 时发生异常: {e}"
else:
    observation = f"错误: 未定义的工具 '{tool_name}'"

observation_str = f"Observation: {observation}"
print(f"{observation_str}\n" + "="*40)
prompt_history.append(observation_str)